# 12. edge-lab 서버를 API 로 부르기 (REST)

edge-lab(`run.bat`)을 켜 두면 모델이 서버에 **한 벌** 올라가 있다. 노트북은 HTTP 로 사진을 보내고 JSON 결과만 받는다.
모델을 직접 올리지 않으니 메모리를 아끼고, 다른 언어·다른 PC 에서도 같은 방식으로 부를 수 있다.

- 주소: `http://127.0.0.1:57711` · 전체 목록: 브라우저에서 `/docs`
- 사진 API: `multipart/form-data` 로 `uploadFile` 필드에 파일, 옵션은 주소 뒤 `?lang=ko` 처럼
- 응답: `{"type", "result": "ok" | "fail", "data", "elapsed_ms", "device"}`

## 1. 서버 확인

In [ ]:
import os
import time
import cv2
import numpy as np
import requests
from PIL import Image
from IPython.display import display

BASE = os.environ.get("EDGE_API", "http://127.0.0.1:57711")
r = requests.get(BASE + "/ready", timeout=3).json()
print("준비:", r.get("ready"), f"({r.get('loaded')}/{r.get('total')})")
info = requests.get(BASE + "/system", timeout=5).json()
print("장치 배정:", info.get("assign"))

## 2. 사진

In [ ]:
def show(bgr, width=720):
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); img.thumbnail((width, width)); display(img)

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(0)
for _ in range(5):
    ok, frame = cap.read()
cap.release()
if not ok:
    frame = cv2.imread("sample.jpg")
show(frame)

## 3. 사진 API 부르기 — 사물 찾기

In [ ]:
def call(path, bgr, **params):
    ok, jpg = cv2.imencode(".jpg", bgr)
    t0 = time.perf_counter()
    r = requests.post(BASE + path, params=params, files={"uploadFile": ("frame.jpg", jpg.tobytes(), "image/jpeg")}, timeout=120)
    r.raise_for_status()
    j = r.json()
    print(f"{path}: {j['result']} · 서버 {j.get('elapsed_ms')} ms · 왕복 {(time.perf_counter() - t0) * 1000:.0f} ms · {j.get('device', '')}")
    return j

j = call("/object/object_search", frame, lang="ko")
view = frame.copy()
for o in j["data"]["object"]:
    x1, y1, x2, y2 = o["box"]
    cv2.rectangle(view, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(view, f"{o['name_en']} {o['score']}", (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    print(o["name"], o["score"])
show(view)

## 4. 다른 API 들

In [ ]:
print(call("/face/face_analyze", frame, lang="ko")["data"])
print(call("/code/barcode", frame)["data"])
print(call("/vlm/look", frame, prompt="사진을 한 문장으로 설명하세요.", lang="ko")["data"])

## 해 볼 것
1. `/docs` 를 열어 API 목록을 보고, 배경 지우기(`/gan/portrait`)를 불러 보자. 결과 사진은 base64 로 온다 —
   `cv2.imdecode(np.frombuffer(base64.b64decode(s), np.uint8), cv2.IMREAD_COLOR)` 로 되돌린다.
2. 02 번 직접 추론과 3번 API 의 결과(상자 수 · 시간)를 비교해 보자.
3. 같은 API 를 10번 부르는 동안 서버 시간과 왕복 시간의 차이는 어디서 생기나?